In [1]:
import os
import pandas as pd
import numpy as np
import time
import json
from scipy.stats import kendalltau

# === Aggregated Average Ranking Algorithm ===
def aggregated_ranking_algorithm(df):
    return df.groupby("book_id")["normalizedOverall"].mean().to_dict()

# === Kendall’s τ computation ===
def compute_kendall_tau(rankings1, rankings2):
    common_items = list(set(rankings1.keys()) & set(rankings2.keys()))
    if len(common_items) < 2:
        return 0
    r1 = [rankings1[i] for i in common_items]
    r2 = [rankings2[i] for i in common_items]
    tau, _ = kendalltau(r1, r2)
    return tau

# === Load Goodreads dataset ===
file_path = "/home/martimsbaltazar/Desktop/tese/datasets/goodreads/goodreads_reviews_spoiler.json"

data = []
with open(file_path, "r") as f:
    for line in f:
        entry = json.loads(line)
        rating = (entry["rating"] - 1) / 4  # normalize to [0,1]
        data.append({
            "user_id": entry["user_id"],
            "book_id": entry["book_id"],
            "normalizedOverall": rating
        })

df = pd.DataFrame(data)

# Compute original rankings
rankings = aggregated_ranking_algorithm(df)

In [2]:
# === Robustness (spam injection) ===
spam_dir = "/home/martimsbaltazar/Desktop/tese/datasets/goodreads/spam_versions_goodreads"
ratios = [10, 30, 50, 70]

print("=== Robustness Analysis (Aggregated Average, Goodreads) ===")
for percent in ratios:
    start_time = time.time()

    file_name = f"reviews_with_{percent}percent_spam.json"
    file_path = os.path.join(spam_dir, file_name)

    
    # Load the dataset
    with open(file_path, 'r') as f:
        data = [json.loads(line) for line in f]
    df_attack = pd.DataFrame(data)

    # Get unique users and items
    users = df_attack["user_id"].unique()
    items = df_attack["book_id"].unique()

    # Normalize the 'rating' column to range [0, 1]
    min_rating = 1
    max_rating = 5
    df_attack["normalizedOverall"] = (df_attack["rating"] - min_rating) / (max_rating - min_rating)

    # Compute rankings using your bipartite ranking algorithm
    rankingsSpam = aggregated_ranking_algorithm(df_attack)

    # Compute Kendall's tau
    tau_value = compute_kendall_tau(rankings, rankingsSpam)

    # Measure elapsed time
    elapsed_time = time.time() - start_time

    # Print the result
    print(f"[{percent}% Spam] Kendall’s τ: {tau_value:.4f} | Time: {elapsed_time:.2f} seconds")



=== Robustness Analysis (Aggregated Average, Goodreads) ===
[10% Spam] Kendall’s τ: 0.9442 | Time: 36.19 seconds
[30% Spam] Kendall’s τ: 0.8884 | Time: 38.43 seconds
[50% Spam] Kendall’s τ: 0.8514 | Time: 37.69 seconds
[70% Spam] Kendall’s τ: 0.8262 | Time: 38.79 seconds
